# 🎯 Módulo 02 — Salida Estructurada (Pydantic)

> **Objetivo:** Obtener respuestas del agente como objetos Python tipados en lugar de texto libre.

## ¿Por qué salida estructurada?

Cuando construyes una aplicación real, no quieres procesar texto libre con regex. Quieres datos validados.
Definimos un modelo Pydantic y el framework configura automáticamente el JSON Schema correspondiente.

```
Texto libre del usuario  ──→  Agente  ──→  JSON validado  ──→  Objeto Pydantic
```

## API en Python

```python
from pydantic import BaseModel

class MyModel(BaseModel):
    field: str

agent = Agent(client=..., default_options={"response_format": MyModel})
response = await agent.run("...")
parsed = MyModel.model_validate_json(response.text)
```

> En C# es `agent.RunAsync<T>(...)`; en Python pasas el modelo en `response_format` y validas con `.model_validate_json()`.


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Análisis de sentimiento (modelo simple)

Definimos un modelo con tres campos. Las anotaciones `Field(description=...)` viajan al esquema JSON y ayudan al modelo a entender qué espera cada campo.


In [ ]:
from pydantic import BaseModel, Field


class SentimentAnalysis(BaseModel):
    sentiment: str = Field(description="Sentimiento detectado: positive, negative, o neutral")
    confidence: float = Field(description="Score de confianza entre 0.0 y 1.0")
    explanation: str = Field(description="Breve explicación de por qué se detectó este sentimiento")


agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un motor de análisis de sentimiento. Analiza el sentimiento del texto "
        "proporcionado y responde SIEMPRE con JSON válido que cumpla el esquema solicitado."
    ),
    default_options={"response_format": SentimentAnalysis},
)

response = await agent.run(
    "I absolutely love this new Agent Framework! It makes building AI agents so much easier and fun!"
)

result = SentimentAnalysis.model_validate_json(response.text or "")

print("✅ Análisis estructurado:")
print(f"   Sentimiento: {result.sentiment}")
print(f"   Confianza:   {result.confidence:.0%}")
print(f"   Explicación: {result.explanation}")


## 2️⃣ Modelo con nesting + arrays

Pydantic soporta sub-modelos y listas. El framework genera el esquema completo automáticamente.


In [ ]:
class AddressInfo(BaseModel):
    city: str
    country: str


class ExtractedPersonInfo(BaseModel):
    name: str
    age: int
    occupation: str
    skills: list[str]
    address: AddressInfo | None = None


agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Extraes información de personas del texto. "
        "Responde SIEMPRE con JSON válido que cumpla el esquema solicitado."
    ),
    default_options={"response_format": ExtractedPersonInfo},
)

text = (
    "Maria Garcia is a 35-year-old software architect from Madrid, Spain. "
    "She specializes in C#, Azure, and AI/ML technologies."
)
response = await agent.run(text)
person = ExtractedPersonInfo.model_validate_json(response.text or "")

print("✅ Información extraída (objeto anidado):")
print(f"   Nombre:      {person.name}")
print(f"   Edad:        {person.age}")
print(f"   Ocupación:   {person.occupation}")
print(f"   Habilidades: {', '.join(person.skills)}")
if person.address:
    print(f"   Ciudad:      {person.address.city}")
    print(f"   País:        {person.address.country}")


## 🎯 Resumen

| Paso | API |
|------|-----|
| Definir esquema | `class MyModel(BaseModel): field: str = Field(description="...")` |
| Configurar agente | `default_options={"response_format": MyModel}` |
| Validar respuesta | `MyModel.model_validate_json(response.text)` |

> 💡 **Tip:** los campos `Field(description=...)` mejoran muchísimo la precisión del modelo porque viajan al JSON Schema.

**Siguiente módulo →** [Módulo 03 — Function Tools](./03_function_tools.ipynb)
